# TT+MLA+GQA — Pure Scaling

GPT-2 50K vocab | 12 layers | FFN 4× (1024) | 22.6M params | <16MB artifact


## 1. Install


In [ ]:
!pip install -q torch transformers tokenizers datasets tqdm numpy
import torch
print(f"PyTorch {torch.__version__} | GPU: {torch.cuda.get_device_name(0)} | VRAM: {torch.cuda.get_device_properties(0).total_mem/1e9:.1f} GB")


## 2. Clone


In [ ]:
!git clone -b tt-mla-gqa-submission https://github.com/ironbyte-rgb/parameter-golf.git /content/parameter-golf
%cd /content/parameter-golf
import sys; sys.path.insert(0, "/content/parameter-golf")


## 3. Verify model


In [ ]:
from model import TTMLATransformer
import torch
m = TTMLATransformer().cuda()
n = sum(p.numel() for p in m.parameters())
print(f"Params: {n:,} ({n*2/1e6:.1f} MB BF16)")
x = torch.randint(0, 50257, (4, 512), device="cuda")
with torch.no_grad():
    loss = m(x, return_loss=True, targets=x.clone())
print(f"Init loss: {loss.item():.1f} (random ~ln(50257)=10.8)")
del m, x; torch.cuda.empty_cache()
print("OK")


## 4. GPT-2 tokenizer + byte LUTs


In [ ]:
from tokenizer_ import get_gpt2_tokenizer, build_byte_luts
tok = get_gpt2_tokenizer(save_path="./tokenizer.json")
luts = build_byte_luts(tok, vocab_size=50257)
print(f"Vocab: {tok.get_vocab_size()}, LUTs ready")


## 5. Pre-tokenize (~10 min, run once)


In [ ]:
!python prepare_data.py --num_tokens 2e9 --tokenizer ./tokenizer.json


## 6. Train


In [ ]:
%cd /content/parameter-golf
import os
os.environ["BATCH_SIZE"] = "512"
os.environ["TRAIN_STEPS"] = "10000"
os.environ["LR"] = "6e-3"
os.environ["WARMUP_STEPS"] = "500"
os.environ["DECAY_STEPS"] = "1000"
os.environ["EVAL_EVERY"] = "1000"
os.environ["MAX_VAL_TOKENS"] = "200000"
os.environ["OUTPUT_DIR"] = "/content/parameter-golf/output"
print(f"Batch: {os.environ["BATCH_SIZE"]} | Tokens/step: {int(os.environ["BATCH_SIZE"]) * 512:,}")
from train_gpt import train, get_config
train(get_config())


## 7. Results


In [ ]:
%cd /content/parameter-golf
import os
for f in sorted(os.listdir("./output")):
    sz = os.path.getsize(f"./output/{f}") / 1e6
    print(f"  {f:40s} {sz:.2f} MB")
ptz = "./output/final_model.int8.ptz"
if os.path.exists(ptz):
    ptz_sz = os.path.getsize(ptz)
    code_sz = os.path.getsize("./train_gpt.py")
    total = ptz_sz + code_sz
    print(f"
Artifact: {total:,} bytes ({total/1e6:.2f} MB)  Under 16MB: {total < 16_000_000}")
